In [ ]:
import pandas as pd
import numpy as np  


In [ ]:
#1. Transform tables from csv files
#WD EID,Emp Name,5/17/2026_Answered,5/17/2026_AHT,5/18/2026_Answered => WD EID	Emp Name	Date	Answered	AHT

df_may = pd.read_csv(r"data/May.csv")

date_cols = [col for col in df_may.columns if "/" in col]

df_long = pd.melt(
    df_may,
    id_vars=["WD EID", "Emp Name"],
    value_vars=date_cols,
    var_name="temp_col",
    value_name="val",
)

df_long[["Date", "Metric"]] = df_long["temp_col"].str.rsplit(
    "_", n=1, expand=True
)
df_long = df_long.drop(columns=["temp_col"])

df_final_5 = df_long.pivot_table(
    index=["WD EID", "Emp Name", "Date"],
    columns="Metric",
    values="val"
).reset_index()


df_final_5.reindex(columns=["WD EID", "Emp Name", "Date", "Answered", "AHT"])

In [ ]:

df_june = pd.read_csv(r"data/June.csv")

date_cols = [col for col in df_june.columns if "/" in col]

df_long = pd.melt(
    df_june,
    id_vars=["WD EID", "Emp Name"],
    value_vars=date_cols,
    var_name="temp_col",
    value_name="val",
)

df_long[["Date", "Metric"]] = df_long["temp_col"].str.rsplit(
    "_", n=1, expand=True
)
df_long = df_long.drop(columns=["temp_col"])

df_final_6 = df_long.pivot_table(
    index=["WD EID", "Emp Name", "Date"],
    columns="Metric",
    values="val"
).reset_index()


df_final_6.reindex(columns=["WD EID", "Emp Name", "Date", "Answered", "AHT"])

In [ ]:
df_july = pd.read_csv(r"data/July.csv")

date_cols = [col for col in df_july.columns if "/" in col]

df_long = pd.melt(
    df_july,
    id_vars=["WD EID", "Emp Name"],
    value_vars=date_cols,
    var_name="temp_col",
    value_name="val",
)

df_long[["Date", "Metric"]] = df_long["temp_col"].str.rsplit(
    "_", n=1, expand=True
)
df_long = df_long.drop(columns=["temp_col"])

df_final_7 = df_long.pivot_table(
    index=["WD EID", "Emp Name", "Date"],
    columns="Metric",
    values="val"
).reset_index()


df_final_7.reindex(columns=["WD EID", "Emp Name", "Date", "Answered", "AHT"])

In [ ]:
#Final file saved in "src/pre_processing.py"

In [ ]:
#2. Process schedule file
# Employee ID,Employee Name,6/6/2026,6/7/2026,6/8/2026,6/9/2026,6/10/2026,6/11/2026 => Employee ID	Employee Name	Month	Year	Shift	shift_count
df = pd.read_csv("data/Work_Schedule(6-7).csv")

date_cols = [col for col in df.columns if "/" in col]

df_long = pd.melt(
    df,
    id_vars=["Employee ID", "Employee Name"],
    value_vars=date_cols,
    var_name="temp_col",
    value_name="val",
)

df_long = df_long.assign(
    Month = df_long.temp_col.str[:1:],
    Year = df_long.temp_col.str[-4::]
)


,Employee ID,Employee Name,temp_col,val,Month,Year
0,102496670,Dieu Tien,6/6/2026,06:00-15:00,6,2026
1,102496681,Nguyen Thi Phuong Thuy Nguyen,6/6/2026,WO-WO,6,2026
2,102845218,Nguyen Ngoc Anh Thu,6/6/2026,11:00-20:00,6,2026
3,102893381,Vu Van Khang,6/6/2026,21:00-06:00,6,2026
4,102972298,Nguyen Thi Bich Ngan,6/6/2026,06:00-15:00,6,2026
...,...,...,...,...,...,...
1503,103455214,Nguyen Ngoc My Kim,8/2/2026,06:00-15:00,8,2026
1504,103456015,Nguyen Ngoc Thanh Tam,8/2/2026,WO-WO,8,2026
1505,103471626,Nguyen Ho Minh Dai,8/2/2026,NaN,8,2026
1506,103477971,Nguyen Ngoc Quynh Giang,8/2/2026,11:00-20:00,8,2026


In [106]:
shift_map = {
    "06:00-15:00": "Day",
    "11:00-20:00": "Noon",
    "12:00-21:00": "Noon",
    "21:00-06:00": "Night"
}

df_long["Shift"] = df_long["val"].map(shift_map)

df_schedule = df_long.reindex(columns=["Employee ID", "Employee Name", "Month", "Year", "Shift"]).sort_values(["Employee ID", "Employee Name", "Month", "Year"])

df_schedule = df_schedule.dropna().groupby(["Employee ID", "Employee Name", "Month", "Year", "Shift"]).agg(
    shift_count = ("Shift", "count")
).reset_index()

df_schedule = df_schedule.sort_values(["Employee ID", "Employee Name", "Month", "Year", "shift_count"], ascending=[True, True, True, True, False]).drop_duplicates(subset=["Employee ID", "Employee Name", "Month", "Year"], keep="first")

df_schedule = df_schedule.astype({"Month": "int16", "Year":"int16"})

df_schedule

,Employee ID,Employee Name,Month,Year,Shift,shift_count
0,102496670,Dieu Tien,6,2026,Day,17
2,102496670,Dieu Tien,7,2026,Noon,17
3,102496670,Dieu Tien,8,2026,Noon,1
4,102496681,Nguyen Thi Phuong Thuy Nguyen,6,2026,Noon,16
5,102496681,Nguyen Thi Phuong Thuy Nguyen,7,2026,Night,20
...,...,...,...,...,...,...
75,103477971,Nguyen Ngoc Quynh Giang,6,2026,Noon,17
76,103477971,Nguyen Ngoc Quynh Giang,7,2026,Noon,20
77,103477971,Nguyen Ngoc Quynh Giang,8,2026,Noon,2
78,103477990,Le Duy Thu Huyen,6,2026,Noon,17


In [ ]:
#3. Merge two tables (Schedule and Performance)
# 	WD EID	Emp Name	Answered	AHT	Month	Year	Dayofweek	Monthly_AHT	Dayofweek_AHT	Monthly_Answered	Dayofweek_Answered	Shift

df_emp = pd.read_csv("data/Cleaned_sheet(pivot_table).csv", parse_dates = True, low_memory = False, index_col = 'Date')

df_emp.index

df_emp = df_emp.assign(
    Month = df_emp.index.month,
    Year = df_emp.index.year,
    Dayofweek = df_emp.index.weekday
)

In [ ]:
df_emp.head()

In [ ]:
df_emp[["AHT", "Answered"]].describe()

In [ ]:
df_emp = df_emp.assign(
    Monthly_AHT = df_emp.groupby(["WD EID", "Emp Name", "Year", "Month"])["AHT"].transform("mean"),
    Dayofweek_AHT = df_emp.groupby(["WD EID", "Emp Name", "Year", "Month", "Dayofweek"])["AHT"].transform("mean"),
    Monthly_Answered = df_emp.groupby(["WD EID", "Emp Name", "Year", "Month"])["Answered"].transform("mean"),
    Dayofweek_Answered = df_emp.groupby(["WD EID", "Emp Name", "Year", "Month", "Dayofweek"])["Answered"].transform("mean"),
)


In [ ]:
df_emp[df_emp["Emp Name"] =="Nguyen Duc Thuan" ]

In [ ]:
#3.1 Merge schedule and emp table

df_insight = pd.merge(df_emp, df_schedule[["Employee ID", "Employee Name", "Year", "Month", "Shift"]], how="left", left_on=["WD EID", "Emp Name", "Year", "Month"], right_on=["Employee ID", "Employee Name", "Year", "Month"])


In [ ]:
#3.2 Assign May's Shift for each employee since schedule data of May is not available in original Work_Schedule(6-7).csv file
day_employees = [
    "Nguyen Y Bao Han", "Dieu Tien", "Nguyen Ngoc Anh Thu", 
    "Le Hau", "Tran Hoang Anh", "Vo Truong Trong Tin"
]

noon_employees = [
    "Vu Van Khang", "Nguyen Thi Bich Ngan", "Vuong Hoang Bao", 
    "Nguyen Cao Huy Hung", "Truong Huynh Bao Han", "Nguyen Tran Anh Tu", 
    "Tran Duc Binh", "Tran Nguyen Hoang Trieu", "Le Phi Hoang", 
    "Cao Minh Hoang", "Nguyen Duc Thuan", "Tran Thanh Viet Ha", 
    "Huynh Le Thuy Thanh", "Nguyen Ngoc My Kim"
]

night_employees = [
    "Truong Thi Thanh Tam", "Nguyen Thuy Duong", "Nguyen Tran Khuong Duy", 
    "Nguyen Ngoc Thanh Tam", "Nguyen Ho Minh Dai", "Nguyen Ngoc Quynh Giang", 
    "Le Duy Thu Huyen", "Nguyen Thi Phuong Thuy Nguyen"
]

cond_day = (df_insight['Month'] == 5) & (df_insight['Shift'].isna()) & (df_insight['Emp Name'].isin(day_employees))
cond_noon = (df_insight['Month'] == 5) & (df_insight['Shift'].isna()) & (df_insight['Emp Name'].isin(noon_employees))
cond_night = (df_insight['Month'] == 5) & (df_insight['Shift'].isna()) & (df_insight['Emp Name'].isin(night_employees))

conditions = [cond_day, cond_noon, cond_night]

choices = ['Day', 'Noon', 'Night']

df_insight['Shift'] = np.select(conditions, choices, default=df_insight['Shift'])



In [111]:
df_insight.head(10)

,WD EID,Emp Name,Answered,AHT,Month,Year,Dayofweek,Monthly_AHT,Dayofweek_AHT,Monthly_Answered,Dayofweek_Answered,Shift
0,102496670,Dieu Tien,60.0,452.0,5,2026,6,452.818182,473.666667,60.181818,57.666667,Day
1,102496670,Dieu Tien,60.0,451.0,5,2026,0,452.818182,476.500000,60.181818,57.500000,Day
2,102496670,Dieu Tien,70.0,414.0,5,2026,1,452.818182,411.500000,60.181818,67.500000,Day
3,102496670,Dieu Tien,70.0,406.0,5,2026,4,452.818182,451.000000,60.181818,63.500000,Day
4,102496670,Dieu Tien,55.0,412.0,5,2026,5,452.818182,441.000000,60.181818,56.000000,Day
5,102496670,Dieu Tien,53.0,521.0,5,2026,6,452.818182,473.666667,60.181818,57.666667,Day
6,102496670,Dieu Tien,55.0,502.0,5,2026,0,452.818182,476.500000,60.181818,57.500000,Day
7,102496670,Dieu Tien,65.0,409.0,5,2026,1,452.818182,411.500000,60.181818,67.500000,Day
8,102496670,Dieu Tien,57.0,496.0,5,2026,4,452.818182,451.000000,60.181818,63.500000,Day
9,102496670,Dieu Tien,57.0,470.0,5,2026,5,452.818182,441.000000,60.181818,56.000000,Day


In [109]:


df_insight.isna().sum()

WD EID                0
Emp Name              0
Answered              0
AHT                   0
Month                 0
Year                  0
Dayofweek             0
Monthly_AHT           0
Dayofweek_AHT         0
Monthly_Answered      0
Dayofweek_Answered    0
Shift                 0
dtype: int64

In [ ]:
df_insight["Emp Name"].value_counts()
#Employees who have a number of working days less than 30 days will be eliminated 

Emp Name
Nguyen Ngoc Thanh Tam            44
Vu Van Khang                     41
Le Hau                           41
Nguyen Y Bao Han                 41
Le Phi Hoang                     40
Nguyen Tran Khuong Duy           40
Nguyen Ngoc Quynh Giang          40
Le Duy Thu Huyen                 40
Nguyen Thi Phuong Thuy Nguyen    39
Nguyen Tran Anh Tu               39
Dieu Tien                        38
Cao Minh Hoang                   38
Huynh Le Thuy Thanh              38
Nguyen Ngoc My Kim               38
Vuong Hoang Bao                  37
Tran Duc Binh                    37
Tran Nguyen Hoang Trieu          37
Nguyen Ho Minh Dai               37
Nguyen Ngoc Anh Thu              36
Truong Thi Thanh Tam             36
Tran Hoang Anh                   36
Tran Thanh Viet Ha               36
Nguyen Duc Thuan                 35
Vo Truong Trong Tin              35
Nguyen Thuy Duong                33
Nguyen Thi Bich Ngan             22
Truong Huynh Bao Han              5
Nguyen Cao Huy Hung

In [127]:
df_insight = df_insight[df_insight.groupby("Emp Name")["Emp Name"].transform("count") >= 30]

In [128]:
df_insight.shape

(952, 12)

In [129]:
df_insight["Emp Name"].value_counts()

Emp Name
Nguyen Ngoc Thanh Tam            44
Vu Van Khang                     41
Le Hau                           41
Nguyen Y Bao Han                 41
Le Phi Hoang                     40
Nguyen Tran Khuong Duy           40
Nguyen Ngoc Quynh Giang          40
Le Duy Thu Huyen                 40
Nguyen Thi Phuong Thuy Nguyen    39
Nguyen Tran Anh Tu               39
Dieu Tien                        38
Cao Minh Hoang                   38
Huynh Le Thuy Thanh              38
Nguyen Ngoc My Kim               38
Vuong Hoang Bao                  37
Tran Duc Binh                    37
Tran Nguyen Hoang Trieu          37
Nguyen Ho Minh Dai               37
Nguyen Ngoc Anh Thu              36
Truong Thi Thanh Tam             36
Tran Hoang Anh                   36
Tran Thanh Viet Ha               36
Nguyen Duc Thuan                 35
Vo Truong Trong Tin              35
Nguyen Thuy Duong                33
Name: count, dtype: int64

In [130]:
df_insight[["AHT", "Month", "Monthly_AHT", "Dayofweek_AHT", "Monthly_Answered", "Dayofweek_Answered"]].describe()

,AHT,Month,Monthly_AHT,Dayofweek_AHT,Monthly_Answered,Dayofweek_Answered
count,952.000000,952.000000,952.000000,952.000000,952.000000,952.000000
mean,456.045168,6.004202,456.045168,456.045168,54.310924,54.310924
std,202.896588,0.752132,128.997307,161.531807,11.768684,17.401177
min,148.000000,5.000000,188.083333,174.000000,24.777778,1.000000
25%,360.750000,5.000000,378.923077,368.333333,46.625000,46.375000
50%,411.000000,6.000000,427.866667,422.000000,56.235294,57.000000
75%,514.000000,7.000000,512.400000,512.125000,62.000000,66.000000
max,3359.000000,7.000000,995.000000,1405.500000,77.727273,97.000000


In [132]:
#Conclusion of Pre_processing:
# - Transform and convert May, June and July Performance files into pivot table indicating precisely speific employee's KPI
# - Enrich df_insight with more handy columns (Dayofweek, Year, Month, Monthly_AHT, Dayofweek_AHT, Monthly_Answered, Dayofweek_Answered)
# - Fill NaN rows in Shift Columns and remove Employee who dropped as shown in the dataframe with working days is less than 30 days

#Export csv file for EDA

df_insight.to_csv("data/EDA.csv", index=False)